In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from scipy.stats import rankdata

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import xgboost as xgb
import lightgbm as lgb

# Dynamic dataset path configuration (supports local execution and Kaggle environments)
if os.path.exists('./Dataset/train.csv'):
    TRAIN = './Dataset/train.csv'
    TEST = './Dataset/test.csv'
elif os.path.exists('/kaggle/input/playground-series-s6e9/train.csv'):
    TRAIN = '/kaggle/input/playground-series-s6e9/train.csv'
    TEST = '/kaggle/input/playground-series-s6e9/test.csv'
else:
    TRAIN = '/kaggle/input/competitions/playground-series-s6e9/train.csv'
    TEST = '/kaggle/input/competitions/playground-series-s6e9/test.csv'

train_df = pd.read_csv(TRAIN)
test_df = pd.read_csv(TEST)

print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")
train_df.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Target variable distribution
sns.countplot(data=train_df, x="Will_Buy_EV", ax=axes[0], palette="viridis")
axes[0].set_title("Target Distribution (Will_Buy_EV)")
axes[0].set_xlabel("Will Buy EV")
axes[0].set_ylabel("Count")
for p in axes[0].patches:
    axes[0].annotate(
        f"{int(p.get_height())}",
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha="center",
        va="bottom",
    )

# 2. Missing values in train dataset
train_missing = train_df.isnull().sum()
train_missing = train_missing[train_missing > 0]
if not train_missing.empty:
    train_missing.plot(kind="barh", ax=axes[1], color="salmon")
    axes[1].set_title("Missing Values in Train")
    axes[1].set_xlabel("Count")
else:
    axes[1].text(0.5, 0.5, "No Missing Values in Train", ha="center", va="center", fontsize=12)
    axes[1].set_title("Missing Values in Train")

# 3. Missing values in test dataset
test_missing = test_df.isnull().sum()
test_missing = test_missing[test_missing > 0]
if not test_missing.empty:
    test_missing.plot(kind="barh", ax=axes[2], color="teal")
    axes[2].set_title("Missing Values in Test")
    axes[2].set_xlabel("Count")
else:
    axes[2].text(0.5, 0.5, "No Missing Values in Test", ha="center", va="center", fontsize=12)
    axes[2].set_title("Missing Values in Test")

plt.tight_layout()
plt.show()

In [ ]:
# 1. Target class counts & ratio
print("--- Target Class Counts ---")
print(train_df["Will_Buy_EV"].value_counts())
print("\n--- Target Class Ratio (%) ---")
print(train_df["Will_Buy_EV"].value_counts(normalize=True) * 100)

# 2. Encode categorical features for correlation assessment
df_encoded = train_df.drop(columns=["id"], errors="ignore").copy()
for col in df_encoded.select_dtypes(include=["object", "string"]).columns:
    df_encoded[col] = df_encoded[col].astype("category").cat.codes

# 3. Correlation with target
target_corr = (
    df_encoded.corr()["Will_Buy_EV"]
    .drop("Will_Buy_EV")
    .sort_values(ascending=False)
)
print("\n--- Correlation with Target (Will_Buy_EV) ---")
print(target_corr)

# 4. Correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(
    df_encoded.corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

### Deadzone Analysis: Identifying Zero & Near-Zero EV Purchase Probability Zones
In tabular synthetic competitions, discovering **hard thresholds** and **deadzones** (where target probability $P(Y=1) \approx 0$) provides critical insights into why tree-based models excel and guides potential post-processing or rule-based features.

Below we analyze the multidimensional purchase rate across **Subsidy_Available**, **Environmental_Concern_Level**, **Annual_Income_USD**, and **City_Type**.

In [ ]:
# =========================================================================
# DEADZONE ANALYSIS: SUBSIDY x ENVIRONMENT x INCOME x CITY
# =========================================================================
eda_df = train_df.copy()
eda_df['Target'] = (eda_df['Will_Buy_EV'] == 'Yes').astype(int)
eda_df['Income_Bracket'] = pd.qcut(
    eda_df['Annual_Income_USD'], 4, labels=['Q1 (Low)', 'Q2 (Mid-Low)', 'Q3 (Mid-High)', 'Q4 (High)']
)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Subsidy x Environmental Concern Level
p1 = eda_df.pivot_table(index='Subsidy_Available', columns='Environmental_Concern_Level', values='Target', aggfunc='mean') * 100
sns.heatmap(p1, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[0, 0], cbar_kws={'label': 'Buy Rate (%)'})
axes[0, 0].set_title('EV Buy Rate (%): Subsidy vs Environmental Concern Level', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Subsidy Available')
axes[0, 0].set_xlabel('Environmental Concern Level')

# 2. Subsidy x Income Bracket
p2 = eda_df.pivot_table(index='Subsidy_Available', columns='Income_Bracket', values='Target', aggfunc='mean', observed=False) * 100
sns.heatmap(p2, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[0, 1], cbar_kws={'label': 'Buy Rate (%)'})
axes[0, 1].set_title('EV Buy Rate (%): Subsidy vs Income Bracket', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Subsidy Available')
axes[0, 1].set_xlabel('Income Bracket')

# 3. Subsidy x City Type
p3 = eda_df.pivot_table(index='Subsidy_Available', columns='City_Type', values='Target', aggfunc='mean') * 100
sns.heatmap(p3, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[1, 0], cbar_kws={'label': 'Buy Rate (%)'})
axes[1, 0].set_title('EV Buy Rate (%): Subsidy vs City Type', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Subsidy Available')
axes[1, 0].set_xlabel('City Type')

# 4. Deep Deadzone Zoom-In: Subsidy == 'No' (Environmental Concern vs Income)
no_sub = eda_df[eda_df['Subsidy_Available'] == 'No'].copy()
p4 = no_sub.pivot_table(index='Environmental_Concern_Level', columns='Income_Bracket', values='Target', aggfunc='mean', observed=False) * 100
sns.heatmap(p4, annot=True, fmt='.3f', cmap='Blues_r', ax=axes[1, 1], cbar_kws={'label': 'Buy Rate (%)'})
axes[1, 1].set_title('CRITICAL DEADZONE: When Subsidy = No (Env Concern vs Income)', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Environmental Concern Level')
axes[1, 1].set_xlabel('Income Bracket')

plt.tight_layout()
plt.show()

# Quantitative Deadzone Summary
print("=" * 70)
print("                   QUANTITATIVE DEADZONE REPORT")
print("=" * 70)

# Deadzone 1: No Subsidy & Lowest Environmental Concern
dz1 = eda_df[(eda_df['Subsidy_Available'] == 'No') & (eda_df['Environmental_Concern_Level'] == 1.0)]
print(f"Deadzone 1 (Subsidy=No & Env_Concern=1.0):")
print(f"  - Population Count : {len(dz1):,} ({len(dz1)/len(eda_df)*100:.2f}% of train set)")
print(f"  - Positive Buyers  : {dz1['Target'].sum()} out of {len(dz1):,}")
print(f"  - Actual Buy Rate  : {dz1['Target'].mean()*100:.4f}% (~0% Deadzone!)\n")

# Deadzone 2: No Subsidy & Low Income (Q1) & Env Concern <= 2.0
dz2 = eda_df[(eda_df['Subsidy_Available'] == 'No') & (eda_df['Income_Bracket'] == 'Q1 (Low)') & (eda_df['Environmental_Concern_Level'] <= 2.0)]
print(f"Deadzone 2 (Subsidy=No & Low Income Q1 & Env_Concern <= 2.0):")
print(f"  - Population Count : {len(dz2):,} ({len(dz2)/len(eda_df)*100:.2f}% of train set)")
print(f"  - Positive Buyers  : {dz2['Target'].sum()} out of {len(dz2):,}")
print(f"  - Actual Buy Rate  : {dz2['Target'].mean()*100:.4f}% (Absolute Zero Deadzone!)\n")

# Deadzone 3: No Subsidy & High Range Anxiety
dz3 = eda_df[(eda_df['Subsidy_Available'] == 'No') & (eda_df['Range_Anxiety_Level'] == 'High')]
print(f"Deadzone 3 (Subsidy=No & Range_Anxiety=High):")
print(f"  - Population Count : {len(dz3):,}")
print(f"  - Positive Buyers  : {dz3['Target'].sum()} out of {len(dz3):,}")
print(f"  - Actual Buy Rate  : {dz3['Target'].mean()*100:.4f}% (Absolute Zero Deadzone!)")
print("=" * 70)

## Feature Engineering Pipeline: Digit Extraction + Frequency Encoding + Triple Target Encoding + Dynamic Pruning

This pipeline builds rich tabular features combining:
1. **Digit Extraction & Smooth Binning**: Unlocks latent patterns and generator artifacts across continuous numericals.
2. **Global Frequency Encoding**: Captures prevalence across the combined Train and Test distributions.
3. **Triple Target Encoding**: Leakage-free out-of-fold target encoding averaged across 3 random seeds to suppress variance.
4. **Dynamic Feature Pruning**: Eliminates constant and perfectly collinear features (`|corr| == 1.0`) inside each CV fold.

### Feature Dropping (Static & Dynamic Pruning)
Used twice:
- **Static Pruning** (prior to CV): Drops invariant columns and perfectly duplicate features.
- **Dynamic Pruning** (inside each CV fold): Cleanses any collinearity or invariant columns induced by fold-specific target encoding.

In [ ]:
NUMERIC_COLS = [
    'Environmental_Concern_Level',
    'Annual_Income_USD',
    'Number_of_Cars_Owned',
    'Age',
    'Charging_Stations_Near_Work',
    'Charging_Stations_Near_Home',
    'Daily_Commute_km',
]

CAT_COLS = [
    'Subsidy_Available',
    'Home_Charging_Possible',
    'Gender',
    'Current_Car_Type',
    'City_Type',
    'Range_Anxiety_Level',
]


def drop_redundant_columns(df: pd.DataFrame, numeric_cols: list):
    """Drops constant columns and perfectly collinear pairs (|corr| == 1.0).
    Returns (pruned_dataframe, dropped_columns_list)."""
    sub = df[numeric_cols]

    # Invariant features (single unique value)
    constant_cols = [c for c in numeric_cols if sub[c].nunique(dropna=False) <= 1]
    remaining = [c for c in numeric_cols if c not in constant_cols]
    dropped = list(constant_cols)

    # Perfectly collinear pairs
    if len(remaining) > 1:
        corr = sub[remaining].corr().abs()
        seen = []
        for col in remaining:
            is_dup = any(np.isclose(corr.loc[col, other], 1.0) for other in seen)
            if is_dup:
                dropped.append(col)
            else:
                seen.append(col)

    return df.drop(columns=dropped), dropped

### Feature Construction: Digits, Smooth Keys, Frequency Encoding & Static Pruning

In [ ]:
def build_features(train: pd.DataFrame, test: pd.DataFrame):
    y = train['Will_Buy_EV'].map({'No': 0, 'Yes': 1}).astype(np.float32)
    train_features = train.drop(columns=['id', 'Will_Buy_EV']).copy()
    test_features = test.drop(columns=['id']).copy()
    n_train = len(train_features)
    combined = pd.concat([train_features, test_features], axis=0, ignore_index=True)

    numeric_cols = NUMERIC_COLS.copy()
    cat_cols = CAT_COLS.copy()

    # Imputation fallback
    combined[numeric_cols] = combined[numeric_cols].fillna(combined[numeric_cols].median(numeric_only=True))
    combined[cat_cols] = combined[cat_cols].fillna('Missing').astype(str)

    new_cols = {}

    # 1. Extract digits from 10^-4 place up to 10^3 place
    digit_cols = []
    for col in numeric_cols:
        shifted = (combined[col] * 10_000).round().astype(np.int64)
        for p in range(-4, 4):
            shift = p + 4
            new_col = f'{col}_digit_{p}'
            new_cols[new_col] = ((shifted // (10 ** shift)) % 10).astype(np.int8)
            digit_cols.append(new_col)

    # Environmental Concern Extremes
    new_cols['is_env_hater'] = (combined['Environmental_Concern_Level'] == 1).astype(np.int8)

    # Smooth Keys (Binned continuous variables)
    binned_cols = []
    for col in ['Annual_Income_USD', 'Daily_Commute_km']:
        for scale, suffix in [(1, 'floor1'), (100, 'floor100'), (1000, 'floor1000')]:
            new_col = f'{col}_{suffix}'
            new_cols[new_col] = np.floor(combined[col] / scale).astype(np.int64)
            binned_cols.append(new_col)

    combined = pd.concat([combined, pd.DataFrame(new_cols, index=combined.index)], axis=1)

    # 2. Key strings for Frequency and Target Encoding
    stringify_cols = numeric_cols + digit_cols + binned_cols
    key_cols_dict = {f'{col}_key': combined[col].astype(str) for col in stringify_cols}
    combined = pd.concat([combined, pd.DataFrame(key_cols_dict, index=combined.index)], axis=1)
    key_cols = list(key_cols_dict.keys())

    all_encoding_cols = cat_cols + key_cols

    # 3. Global Frequency Encoding
    freq_cols_dict = {}
    for col in all_encoding_cols:
        freq_map = combined[col].value_counts(normalize=True) * 100
        freq_cols_dict[f'{col}_freq'] = combined[col].map(freq_map).astype(np.float32)
    combined = pd.concat([combined, pd.DataFrame(freq_cols_dict, index=combined.index)], axis=1)
    freq_cols = list(freq_cols_dict.keys())

    # 4. Static Feature Pruning (prior to CV)
    numeric_feature_cols = numeric_cols + digit_cols + binned_cols + freq_cols + ['is_env_hater']
    combined, dropped = drop_redundant_columns(combined, numeric_feature_cols)
    print(f"[Static Pruning] Dropped {len(dropped)}/{len(numeric_feature_cols)} redundant columns.")
    numeric_feature_cols = [c for c in numeric_feature_cols if c not in dropped]

    X = combined.iloc[:n_train].reset_index(drop=True)
    X_test = combined.iloc[n_train:].reset_index(drop=True)

    meta = {
        'numeric_feature_cols': numeric_feature_cols,
        'te_cols': all_encoding_cols,
    }
    return X, y, X_test, meta


X, y, X_test, meta = build_features(train_df, test_df)
print(f"X shape: {X.shape}, X_test shape: {X_test.shape}")
print(f"Direct numeric features: {len(meta['numeric_feature_cols'])}")
print(f"Features for Target Encoding: {len(meta['te_cols'])}")

### Triple Target Encoding (Leakage-Free)
Target encoding is computed strictly Out-Of-Fold (OOF) on the outer training fold using an inner Stratified K-Fold. To minimize variance, the inner OOF encoding is repeated 3 times across different random seeds and averaged ("Triple TE"). Validation and test sets are transformed using statistics derived from the complete outer training split.

In [ ]:
def target_encode_oof_train(train_keys, y_train, n_splits, smoothing, seed):
    """OOF Target Encoding for training data with inner K-Fold cross-validation."""
    global_mean = y_train.mean()
    oof = np.zeros(len(train_keys), dtype=np.float64)
    skf_inner = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in skf_inner.split(train_keys, y_train):
        keys_tr, y_tr = train_keys.iloc[tr_idx], y_train.iloc[tr_idx]
        stats = y_tr.groupby(keys_tr).agg(['mean', 'count'])
        smooth = (stats['mean'] * stats['count'] + global_mean * smoothing) / (stats['count'] + smoothing)
        oof[val_idx] = train_keys.iloc[val_idx].map(smooth).fillna(global_mean).values
    return oof


def target_encode_apply(keys, train_keys, y_train, smoothing):
    """Encodes validation/test data using statistics computed on the complete training split."""
    global_mean = y_train.mean()
    stats = y_train.groupby(train_keys).agg(['mean', 'count'])
    smooth = (stats['mean'] * stats['count'] + global_mean * smoothing) / (stats['count'] + smoothing)
    return keys.map(smooth).fillna(global_mean).values


def triple_target_encode(X_train, X_val, X_test, y_train, cols, n_repeats=3, n_splits=5, smoothing=20):
    """'Triple TE': repeats OOF target encoding 3 times with varied seeds to stabilize training estimates."""
    y_train = y_train.reset_index(drop=True)
    te_train, te_val, te_test = {}, {}, {}

    for col in cols:
        train_keys = X_train[col].reset_index(drop=True)
        val_keys = X_val[col].reset_index(drop=True)
        test_keys = X_test[col].reset_index(drop=True)

        oof_runs = [
            target_encode_oof_train(train_keys, y_train, n_splits, smoothing, seed=100 * r + 42)
            for r in range(n_repeats)
        ]
        te_train[f'{col}_te'] = np.mean(oof_runs, axis=0).astype(np.float32)

        enc = target_encode_apply(
            pd.concat([val_keys, test_keys], ignore_index=True), train_keys, y_train, smoothing
        )
        te_val[f'{col}_te'] = enc[:len(val_keys)].astype(np.float32)
        te_test[f'{col}_te'] = enc[len(val_keys):].astype(np.float32)

    te_train_df = pd.DataFrame(te_train, index=X_train.index)
    te_val_df = pd.DataFrame(te_val, index=X_val.index)
    te_test_df = pd.DataFrame(te_test, index=X_test.index)
    return te_train_df, te_val_df, te_test_df

### Tabular ResNet Architecture (PyTorch)
A deep residual neural network tailored for tabular data: `Linear -> BatchNorm -> SiLU -> Dropout -> Linear -> Residual Connection`.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
torch.manual_seed(42)


class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.BatchNorm1d(dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
        )

    def forward(self, x):
        return x + self.block(x)


class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_blocks=3, dropout=0.2):
        super().__init__()
        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU()
        )
        self.res_blocks = nn.ModuleList([
            ResBlock(hidden_dim, dropout=dropout) for _ in range(num_blocks)
        ])
        self.head = nn.Sequential(
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        x = self.input_layer(x)
        for block in self.res_blocks:
            x = block(x)
        return self.head(x).squeeze(-1)

### Single-Fold Training Function (AdamW + CosineAnnealingLR)
Uses **AdamW** for decoupled weight decay and **CosineAnnealingLR** for smooth learning rate scheduling. Early stopping monitors Validation ROC-AUC.

In [ ]:
def train_one_fold(
    X_train, y_train, X_val, y_val, X_test,
    input_dim,
    epochs=100,
    batch_size=1024,
    lr=1e-3,
    weight_decay=1e-3,
    patience=12,
    verbose_every=5,
):
    train_ds = TensorDataset(
        torch.from_numpy(X_train), torch.from_numpy(y_train.astype(np.float32))
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True)

    X_val_t = torch.from_numpy(X_val).to(device)
    y_val_t = torch.from_numpy(y_val.astype(np.float32)).to(device)
    X_test_t = torch.from_numpy(X_test).to(device)

    model = MLP(input_dim=input_dim).to(device)
    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-6
    )

    best_auc = -np.inf
    best_val_logits = None
    best_test_logits = None
    epochs_no_improve = 0

    for epoch in range(1, epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_loss = criterion(val_logits, y_val_t).item()
            val_probs = torch.sigmoid(val_logits).cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if epoch % verbose_every == 0 or epoch == 1:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"  [MLP] Epoch {epoch:3d} | LR {current_lr:.6f} | Val Loss {val_loss:.5f} | Val AUC {val_auc:.5f}")

        if val_auc > best_auc:
            best_auc = val_auc
            epochs_no_improve = 0
            with torch.no_grad():
                best_val_logits = val_logits.clone()
                best_test_logits = model(X_test_t)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  [MLP] Early stopping at epoch {epoch} (Best Val AUC: {best_auc:.5f})")
                break

    val_preds = torch.sigmoid(best_val_logits).cpu().numpy()
    test_preds = torch.sigmoid(best_test_logits).cpu().numpy()
    return val_preds, test_preds, best_auc

### 5-Fold Cross-Validation & Multi-Model Ensemble (XGBoost + LightGBM + MLP)
Each fold executes:
1. **Fold-specific Triple Target Encoding** & **Dynamic Pruning**.
2. **XGBoost Classifier** (GPU-accelerated histogram boosting).
3. **LightGBM Classifier** (Fast leaf-wise tree boosting).
4. **MLP ResNet** (Deep representation learning with AdamW & Cosine Annealing).
5. **Rank Averaging Blending**: Rescales predicted probabilities to rank percentiles before blending, eliminating calibration discrepancies across model paradigms.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

n_train = len(X)
n_test = len(X_test)

# Out-Of-Fold (OOF) Prediction holders
oof_mlp = np.zeros(n_train)
oof_xgb = np.zeros(n_train)
oof_lgb = np.zeros(n_train)

# Test Prediction accumulators
test_mlp = np.zeros(n_test)
test_xgb = np.zeros(n_test)
test_lgb = np.zeros(n_test)

fold_aucs_mlp = []
fold_aucs_xgb = []
fold_aucs_lgb = []

xgb_device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Starting Ensemble Training (XGBoost + LightGBM + MLP) | PyTorch Device: {device} | XGB Device: {xgb_device}")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n===================== FOLD {fold + 1}/5 =====================")
    X_train_fold = X.iloc[train_idx].copy()
    X_val_fold = X.iloc[val_idx].copy()
    X_test_fold = X_test.copy()
    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx].values

    # --- 1. Triple Target Encoding ---
    te_train, te_val, te_test = triple_target_encode(
        X_train_fold, X_val_fold, X_test_fold, y_train_fold,
        cols=meta['te_cols'], n_repeats=3, n_splits=5, smoothing=20,
    )

    X_train_final = pd.concat([
        X_train_fold[meta['numeric_feature_cols']].reset_index(drop=True),
        te_train.reset_index(drop=True),
    ], axis=1)
    X_val_final = pd.concat([
        X_val_fold[meta['numeric_feature_cols']].reset_index(drop=True),
        te_val.reset_index(drop=True),
    ], axis=1)
    X_test_final = pd.concat([
        X_test_fold[meta['numeric_feature_cols']].reset_index(drop=True),
        te_test.reset_index(drop=True),
    ], axis=1)

    # --- 2. Dynamic Pruning ---
    X_train_final, dropped_dyn = drop_redundant_columns(X_train_final, X_train_final.columns.tolist())
    X_val_final = X_val_final.drop(columns=dropped_dyn)
    X_test_final_fold = X_test_final.drop(columns=dropped_dyn)
    print(f"  [Dynamic Pruning] Fold {fold + 1}: Dropped {len(dropped_dyn)} columns -> {X_train_final.shape[1]} features remaining")

    # =========================================================================
    # MODEL 1: XGBoost (Tree-based SOTA)
    # =========================================================================
    print(f"\n  >>> Training XGBoost (Fold {fold + 1})...")
    xgb_model = xgb.XGBClassifier(
        n_estimators=2200,
        learning_rate=0.028,
        max_depth=6,
        subsample=0.85,
        colsample_bytree=0.70,
        reg_alpha=0.5,
        reg_lambda=3.0,
        tree_method='hist',
        device=xgb_device,
        eval_metric='auc',
        early_stopping_rounds=50,
        random_state=42 + fold,
    )
    xgb_model.fit(
        X_train_final, y_train_fold,
        eval_set=[(X_val_final, y_val_fold)],
        verbose=100,
    )
    xgb_val_pred = xgb_model.predict_proba(X_val_final)[:, 1]
    xgb_test_pred = xgb_model.predict_proba(X_test_final_fold)[:, 1]
    xgb_auc = roc_auc_score(y_val_fold, xgb_val_pred)
    oof_xgb[val_idx] = xgb_val_pred
    test_xgb += xgb_test_pred / skf.n_splits
    fold_aucs_xgb.append(xgb_auc)
    print(f"  --> XGBoost Fold {fold + 1} Best AUC: {xgb_auc:.5f}")

    # =========================================================================
    # MODEL 2: LightGBM (Fast Gradient Boosting)
    # =========================================================================
    print(f"\n  >>> Training LightGBM (Fold {fold + 1})...")
    lgb_model = lgb.LGBMClassifier(
        n_estimators=2500,
        learning_rate=0.035,
        num_leaves=63,
        max_depth=-1,
        subsample=0.82,
        subsample_freq=1,
        colsample_bytree=0.65,
        reg_alpha=0.45,
        reg_lambda=2.80,
        metric='auc',
        random_state=42 + fold,
        n_jobs=-1,
        verbose=-1,
    )
    lgb_model.fit(
        X_train_final, y_train_fold,
        eval_set=[(X_val_final, y_val_fold)],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    lgb_val_pred = lgb_model.predict_proba(X_val_final)[:, 1]
    lgb_test_pred = lgb_model.predict_proba(X_test_final_fold)[:, 1]
    lgb_auc = roc_auc_score(y_val_fold, lgb_val_pred)
    oof_lgb[val_idx] = lgb_val_pred
    test_lgb += lgb_test_pred / skf.n_splits
    fold_aucs_lgb.append(lgb_auc)
    print(f"  --> LightGBM Fold {fold + 1} Best AUC: {lgb_auc:.5f}")

    # =========================================================================
    # MODEL 3: MLP (Tabular ResNet + AdamW + CosineAnnealing)
    # =========================================================================
    print(f"\n  >>> Training MLP ResNet (Fold {fold + 1})...")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_final).astype(np.float32)
    X_val_scaled = scaler.transform(X_val_final).astype(np.float32)
    X_test_scaled = scaler.transform(X_test_final_fold).astype(np.float32)

    mlp_val_pred, mlp_test_pred, mlp_auc = train_one_fold(
        X_train_scaled, y_train_fold.values, X_val_scaled, y_val_fold, X_test_scaled,
        input_dim=X_train_scaled.shape[1],
        epochs=80,
        batch_size=1024,
        lr=1e-3,
        weight_decay=1e-3,
        patience=12,
    )
    oof_mlp[val_idx] = mlp_val_pred
    test_mlp += mlp_test_pred / skf.n_splits
    fold_aucs_mlp.append(mlp_auc)
    print(f"  --> MLP Fold {fold + 1} Best AUC: {mlp_auc:.5f}")

# =========================================================================
# OUT-OF-FOLD (OOF) EVALUATION & ENSEMBLE BLENDING
# =========================================================================
cv_xgb = roc_auc_score(y, oof_xgb)
cv_lgb = roc_auc_score(y, oof_lgb)
cv_mlp = roc_auc_score(y, oof_mlp)

print("\n" + "=" * 60)
print("             OUT-OF-FOLD (OOF) ROC-AUC RESULTS")
print("=" * 60)
print(f"1. XGBoost OOF ROC-AUC    : {cv_xgb:.5f}")
print(f"2. LightGBM OOF ROC-AUC  : {cv_lgb:.5f}")
print(f"3. MLP ResNet OOF ROC-AUC : {cv_mlp:.5f}")

# Rank Averaging Ensemble
rank_xgb = rankdata(oof_xgb) / n_train
rank_lgb = rankdata(oof_lgb) / n_train
rank_mlp = rankdata(oof_mlp) / n_train

oof_ensemble = 0.40 * rank_xgb + 0.40 * rank_lgb + 0.20 * rank_mlp
cv_ensemble = roc_auc_score(y, oof_ensemble)

print("-" * 60)
print(f"---> OVERALL ENSEMBLE (XGB + LGB + MLP) ROC-AUC: {cv_ensemble:.5f} <---")
print("=" * 60)

# =========================================================================
# GENERATE SUBMISSION FILES
# =========================================================================
pd.DataFrame({'id': test_df['id'], 'Will_Buy_EV': test_xgb}).to_csv('submission_xgb.csv', index=False)
pd.DataFrame({'id': test_df['id'], 'Will_Buy_EV': test_lgb}).to_csv('submission_lgb.csv', index=False)
pd.DataFrame({'id': test_df['id'], 'Will_Buy_EV': test_mlp}).to_csv('submission_mlp.csv', index=False)

# Submission Ensemble (Rank Averaging)
test_rank_xgb = rankdata(test_xgb) / n_test
test_rank_lgb = rankdata(test_lgb) / n_test
test_rank_mlp = rankdata(test_mlp) / n_test
final_ensemble_test = 0.40 * test_rank_xgb + 0.40 * test_rank_lgb + 0.20 * test_rank_mlp

sub_ensemble = pd.DataFrame({'id': test_df['id'], 'Will_Buy_EV': final_ensemble_test})
sub_ensemble.to_csv('submission_ensemble.csv', index=False)

print("Saved all 4 submission files:")
print(" - submission_xgb.csv")
print(" - submission_lgb.csv")
print(" - submission_mlp.csv")
print(" - submission_ensemble.csv (Recommended submission for best competition score)")